Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  #  local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

Imports

In [2]:
import torch
import json
import random
from tqdm import tqdm
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


load Model

In [3]:

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3.1-8b-Instruct",   # Base model
    max_seq_length=max_seq_length,
    dtype=torch.float16,
    load_in_4bit=True  # REQUIRED if using T4 GPU
)

FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((409

Load Your Dataset

In [4]:
data = []

with open("OS-cleaned_file.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

print("Total samples:", len(data))

# Shuffle to avoid bias
random.shuffle(data)

Total samples: 1465


Create 3-Shot Split

In [5]:
few_shot_examples = data[:3]
test_data = data[3:]

print("Few-shot examples:", len(few_shot_examples))
print("Test samples:", len(test_data))

Few-shot examples: 3
Test samples: 1462


Build 3-Shot Prompt

In [6]:
def build_3shot_prompt(instruction, input_text):
    few_shot_block = ""

    for ex in few_shot_examples:
        few_shot_block += f"""### Instruction:
{ex["instruction"]}

### Input:
{ex["input"]}

### Response:
{ex["output"]}

"""

    final_prompt = f"""{few_shot_block}
### Instruction:
{instruction}

### Input:
{input_text}

### Rules:
- Return valid JSON only.
- Do NOT repeat entities.
- Use only the allowed labels.
- If no entity exists, return [].

### Response:
"""

    return final_prompt

Run Inference

In [7]:

predictions = []

for sample in tqdm(test_data):

    prompt_text = build_3shot_prompt(
        sample["instruction"],
        sample["input"]
    )

    # IMPORTANT: Use chat template for base model
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "You are an expert entity extraction system. Return valid JSON only."},
            {"role": "user", "content": prompt_text}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.0,   # deterministic for structured output
        top_p=1.0,
        do_sample=False
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    prediction = decoded.split("assistant")[-1].strip()

    predictions.append({
        "ground_truth": sample["output"],
        "prediction": prediction
    })

100%|██████████| 1462/1462 [3:55:11<00:00,  9.65s/it]


Evaluation

In [10]:
import json
from collections import defaultdict

def extract_entities(output):
    try:
        parsed = json.loads(output)
        return set((d["entity"], d["label"]) for d in parsed)
    except:
        return set()

# -------- Micro counts --------
tp = 0
fp = 0
fn = 0

# -------- Macro F1 storage --------
per_sample_f1 = []

# -------- Partial accuracy storage --------
partial_scores = []

# -------- Exact Match Accuracy --------
correct_samples = 0
total_samples = 0

for p in predictions:
    gt = extract_entities(p["ground_truth"])
    pred = extract_entities(p["prediction"])

    total_samples += 1

    # ---- Exact Match Accuracy ----
    if gt == pred:
        correct_samples += 1

    # ---- Micro ----
    tp_sample = len(gt & pred)
    fp_sample = len(pred - gt)
    fn_sample = len(gt - pred)

    tp += tp_sample
    fp += fp_sample
    fn += fn_sample

    # ---- Per-sample Precision / Recall / F1 (for Macro F1) ----
    precision_sample = tp_sample / (tp_sample + fp_sample + 1e-9)
    recall_sample = tp_sample / (tp_sample + fn_sample + 1e-9)
    f1_sample = 2 * precision_sample * recall_sample / (precision_sample + recall_sample + 1e-9)

    per_sample_f1.append(f1_sample)

    # ---- Partial Accuracy (Jaccard Similarity) ----
    union = len(gt | pred)
    intersection = len(gt & pred)
    partial_score = intersection / (union + 1e-9)
    partial_scores.append(partial_score)

# ==============================
# Final Metrics
# ==============================

# ---- Micro ----
micro_precision = tp / (tp + fp + 1e-9)
micro_recall = tp / (tp + fn + 1e-9)
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall + 1e-9)

# ---- Macro ----
macro_f1 = sum(per_sample_f1) / len(per_sample_f1)

# ---- Partial Accuracy ----
partial_accuracy = sum(partial_scores) / len(partial_scores)

# ---- Exact Match Accuracy ----
accuracy = correct_samples / total_samples

# ==============================
# Print Results
# ==============================

print("\n===== RESULTS =====")
print("Micro Precision :", round(micro_precision, 4))
print("Micro Recall    :", round(micro_recall, 4))
print("Micro F1        :", round(micro_f1, 4))
print("Macro F1        :", round(macro_f1, 4))
print("Partial Accuracy:", round(partial_accuracy, 4))
print("Exact Accuracy  :", round(accuracy, 4))


===== RESULTS =====
Micro Precision : 0.1977
Micro Recall    : 0.3191
Micro F1        : 0.2442
Macro F1        : 0.2268
Partial Accuracy: 0.1421
Exact Accuracy  : 0.0007
